In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

In [ ]:
epochs = 10
data = pd.read_csv("creditcard.csv")
x = data.drop(columns=["Time", "Class"])
y = data["Class"]
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)
x_train, x_val = train_test_split(x_train[y_train == 0], test_size=0.2, random_state=42)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype("float32")
x_val = scaler.transform(x_val).astype("float32")
x_test = scaler.transform(x_test).astype("float32")

In [ ]:
inputs = keras.Input(shape=(x_train.shape[1],))
encoded = keras.layers.Dense(14, activation="tanh")(inputs)
encoded = keras.layers.Dense(7, activation="relu")(encoded)
encoder = keras.Model(inputs, encoded, name="encoder")


In [ ]:
latent_inputs = keras.Input(shape=(7,))
decoded = keras.layers.Dense(7, activation="tanh")(latent_inputs)
decoded = keras.layers.Dense(x_train.shape[1])(decoded)
decoder = keras.Model(latent_inputs, decoded, name="decoder")

In [ ]:
autoencoder = keras.Model(inputs, decoder(encoder(inputs)), name="autoencoder")
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.fit(
    x_train, x_train, epochs=epochs, batch_size=32,
    validation_data=(x_val, x_val),
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
)


In [ ]:
val_error = np.mean((x_val - autoencoder.predict(x_val, verbose=0)) ** 2, axis=1)
threshold = np.quantile(val_error, 0.99)
test_error = np.mean((x_test - autoencoder.predict(x_test, verbose=0)) ** 2, axis=1)
y_pred = (test_error > threshold).astype(int)
print(f"Threshold: {threshold:.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, y_pred, zero_division=0):.4f}")
print("Confusion matrix (normal, fraud):\n", confusion_matrix(y_test, y_pred, labels=[0, 1]))